# Milestone 1 lab, notebook 8 of 8: The production version

## Where we are

| Notebook | What you do |
|---|---|
| 1. The question and five green checks | Meet Lotus Retails and the Monday question. Run the given pipeline for one night. Read its five checks |
| 2. Airflow runs it | Let Airflow run the same night. Read the DAG: its tasks and its arrows |
| 3. The number by hand | Add up one store's receipts yourself. Find out why the two numbers differ |
| 4. Who is hurt | For each table: who uses it, and what happens to them when it is wrong |
| 5. The check card | Meet the page you fill in before any check. Say what one row means |
| 6. Read the table first | Count the rows, find the smallest and largest values, count the empty cells |
| 7. Six questions for any table | Sort your worries under six questions. See which ones the five checks answer |
| **8. The production version** (you are here) | The same logic as a tested program |

**About five minutes.**

In [1]:
import subprocess
import sys
from datetime import date

from lab_helpers import MILESTONE_DIR, load_result, money
from lotus_pipeline import by_hand, pipeline
from lotus_pipeline.warehouse import connect

## Step 1: From a lab to a program

A notebook is a good place to find something out. It is a bad place to keep it: nobody can test a notebook, and nobody runs one at 02:00.

So the three things you did by hand are also in the production code, in `lotus_pipeline/`. Each is one small file with tests.

| You did, in a notebook | The program | File |
|---|---|---|
| Added up one store's file (3) | `by-hand` | `by_hand.py` |
| Asked who is hurt (4) | `impact` | `consumers.py` |
| Read the table first (6) | `profile` | `profile.py` |

In a terminal, from this milestone's folder, you type `uv run python -m lotus_pipeline by-hand --store S03 --day 2026-08-24`. Here the same command is called from Python.

## Step 2: The number by hand

In [2]:
_ = pipeline.main(["by-hand", "--store", "S03", "--day", "2026-08-24"])

S03 on 2026-08-24
  the pipeline says        $796.18
  by hand, from file       $777.20
  the tills' own total     $777.20
  difference                $18.98
  look at line S03-0824-000008: a refund of $9.49, and the amount is not negative


It prints the three counts, the difference, and the row to look at. The same as notebook 3.

In [3]:
yours = load_result("answer_key_by_hand")
with connect() as wh:
    programs = by_hand.compare(wh, "S03", date(2026, 8, 24))

print("Your notebook:", money(yours["by_hand_cents"]), "by hand,", money(yours["pipeline_cents"]), "from the pipeline")
print("The program:  ", money(programs["by_hand_cents"]), "by hand,", money(programs["pipeline_cents"]), "from the pipeline")
print("Equal:", (yours["by_hand_cents"], yours["pipeline_cents"]) == (programs["by_hand_cents"], programs["pipeline_cents"]))

Your notebook: $777.20 by hand, $796.18 from the pipeline
The program:   $777.20 by hand, $796.18 from the pipeline
Equal: True


## Step 3: Who is hurt

In [4]:
_ = pipeline.main(["impact"])

The seven tables, the most harmful first:
  1. sales_lines     CRITICAL  reaches: Store managers, Head of retail operations, The replenishment team, Finance
  2. products        CRITICAL  reaches: Store managers, Head of retail operations, The replenishment team, Finance
  3. stores          CRITICAL  reaches: Store managers, Head of retail operations, The replenishment team, Finance
  4. stock_counts              reaches: The replenishment team, Finance
  5. supplier_prices           reaches: The replenishment team, Finance
  6. members                   reaches: Marketing
  7. till_totals               reaches: nobody yet


## Step 4: Read the table first

In [5]:
_ = pipeline.main(["profile", "--table", "clean.sales_lines"])

Profile of clean.sales_lines: 20,482 rows

  column                   empty  empty %  different  smallest .. largest
  line_id                      0     0.0%     20,482  S01-0727-000001 .. S06-0824-000098
  receipt_id                   0     0.0%      6,166  S01-T1-004001 .. S06-T2-004459
  line_no                      0     0.0%          6  1 .. 6
  store_id                     0     0.0%          6  S01 .. S06
  store_name                   0     0.0%          6  Lotus Ashby High Street .. Lotus Fenwick Harbour
  till_id                      0     0.0%          2  T1 .. T2
  sold_at                      0     0.0%      6,148  2026-07-27 07:27:32 .. 2026-08-24 21:54:59
  sales_date                   0     0.0%         29  2026-07-27 .. 2026-08-24
  product_id                   0     0.0%         40  P-001 .. P-040
  product_name                 0     0.0%         40  Apples .. Whole milk 2L
  category                     0     0.0%          8  Bakery .. Pantry
  unit                 

## Step 5: The tests

One rule in this project is strict: **every check has two tests**. One where its fault is planted and the check fails. One on clean data where it stays silent.

A check that never fails in a test might never fail in real life either. A check that fails on clean data is a false alarm, and people switch those off.

Here are the tests of the five standard checks.

In [6]:
done = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_checks.py", "-v", "--no-header", "--color=no", "-p", "no:cacheprovider"],
    cwd=MILESTONE_DIR, capture_output=True, text=True,
)
print("\n".join(line.split("::")[-1] for line in done.stdout.splitlines() if "::" in line))
print(done.stdout.strip().splitlines()[-1])

test_every_check_is_silent_on_clean_data[1] PASSED [  8%]
test_every_check_is_silent_on_clean_data[2] PASSED [ 16%]
test_every_check_is_silent_on_clean_data[3] PASSED [ 25%]
test_every_check_is_silent_on_clean_data[4] PASSED [ 33%]
test_every_check_is_silent_on_clean_data[5] PASSED [ 41%]
test_check_1_fails_when_the_day_has_no_lines PASSED [ 50%]
test_check_2_fails_on_a_null_key PASSED            [ 58%]
test_check_3_fails_on_a_key_stored_twice PASSED    [ 66%]
test_check_4_fails_on_a_status_that_is_not_allowed PASSED [ 75%]
test_check_5_fails_when_a_stores_day_is_not_above_zero PASSED [ 83%]
test_what_the_five_checks_do_not_catch PASSED      [ 91%]
test_a_failed_check_stops_the_run_before_publish PASSED [100%]
============================== 12 passed in 0.81s ==============================


Read the names. One of them is `test_what_the_five_checks_do_not_catch`. It plants the refund with a positive amount, and proves that all five checks stay silent. That is this milestone, as a test.

To run every test of the milestone, type `uv run pytest` in this folder.

## Step 6: What is still wrong

Look back at what you have.

- A wrong number, and the row that caused it.
- The seven tables, ranked by harm.
- A check card for `sales_lines`, with five questions answered.
- Ten candidate checks, sorted under six questions.

And look at what you do not have. The pipeline is exactly as it was. Tomorrow night the same refund would pass again.

> **We found one error by luck, and we have a list of worries. We have no checks that would have caught it.**

Milestone 2 writes them: through three lenses, by hand first, each with the fault it catches and the one it misses.

## Before you go on

- Open [milestone_1_flow.html](https://vijaysngrajput.github.io/quality-gates-on-a-running-pipeline/src/m1_green_checks_wrong_number/milestone_1_flow.html) in a browser. Watch the refund walk through five green checks.
- Open [milestone_1_recap.html](https://vijaysngrajput.github.io/quality-gates-on-a-running-pipeline/src/m1_green_checks_wrong_number/milestone_1_recap.html) for a look back, and three questions to test yourself.
- Stop what you started: see "Shut it down" in the [README](../README.md#shut-it-down).